In [2]:
!pip install -q mne tensorflow scikit-learn pandas matplotlib

In [3]:
from google.colab import files
import zipfile, shutil
from pathlib import Path

uploaded = files.upload()

zip_files = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
if len(zip_files) == 0:
    raise ValueError("Upload your EEG ZIP file")

ZIP_PATH = zip_files[0]
EXTRACT_DIR = Path("/content/eeg_data")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("ZIP extracted successfully")

Saving Madhav_Reddy_eeg_data.zip to Madhav_Reddy_eeg_data.zip
ZIP extracted successfully


In [4]:
import pandas as pd

edf_files = list(EXTRACT_DIR.rglob("*.edf"))
records = []

labels_csv = list(EXTRACT_DIR.rglob("labels.csv"))

if labels_csv:
    labels_df = pd.read_csv(labels_csv[0])

    for edf in edf_files:
        match = labels_df[labels_df["file_name"].astype(str) == edf.name]

        if len(match) > 0:
            label = int(match.iloc[0]["label"])
            class_name = match.iloc[0]["class_name"]
        else:
            name = edf.name.lower()
            parent = edf.parent.name.lower()

            if "ec" in name or "eyes" in name or "closed" in parent:
                label = 0
                class_name = "Eyes_Closed"
            elif "task" in name or "task" in parent:
                label = 1
                class_name = "Task"
            else:
                label = -1
                class_name = "Unknown"

        records.append([str(edf), edf.name, class_name, label])

else:
    for edf in edf_files:
        name = edf.name.lower()
        parent = edf.parent.name.lower()

        if "ec" in name or "eyes" in name or "closed" in parent:
            label = 0
            class_name = "Eyes_Closed"
        elif "task" in name or "task" in parent:
            label = 1
            class_name = "Task"
        else:
            label = -1
            class_name = "Unknown"

        records.append([str(edf), edf.name, class_name, label])

file_labels = pd.DataFrame(
    records,
    columns=["file_path", "file_name", "class_name", "label"]
)

file_labels = file_labels[file_labels["label"] != -1].reset_index(drop=True)

print(file_labels)
print(file_labels["label"].value_counts())

                                           file_path  \
0  /content/eeg_data/Madhav_Reddy_eeg_data/Task/M...   
1  /content/eeg_data/Madhav_Reddy_eeg_data/Eyes_C...   
2  /content/eeg_data/Madhav_Reddy_eeg_data/Eyes_C...   

                                   file_name   class_name  label  
0  Madhav reddy 01.000.03 AGE 21  TASK_1.edf         Task      1  
1      Madhav reddy 01.000.02 AGE 21  EC.edf  Eyes_Closed      0  
2      Madhav reddy 01.000.04 AGE 21  EC.edf  Eyes_Closed      0  
label
0    2
1    1
Name: count, dtype: int64


In [5]:
import mne
import numpy as np
from sklearn.preprocessing import StandardScaler

WINDOW_SECONDS = 2
OVERLAP_SECONDS = 1
LOW_FREQ = 0.5
HIGH_FREQ = 40
NOTCH_FREQ = 50

X_windows = []
y_windows = []

target_sfreq = None
target_channels = None

for row in file_labels.itertuples():
    raw = mne.io.read_raw_edf(row.file_path, preload=True, verbose=False)

    try:
        raw.pick_types(eeg=True, exclude=[])
    except:
        pass

    raw.filter(LOW_FREQ, HIGH_FREQ, verbose=False)

    try:
        raw.notch_filter(NOTCH_FREQ, verbose=False)
    except:
        pass

    sfreq = int(raw.info["sfreq"])

    if target_sfreq is None:
        target_sfreq = sfreq
    elif sfreq != target_sfreq:
        raw.resample(target_sfreq, verbose=False)

    data = raw.get_data()

    if target_channels is None:
        target_channels = data.shape[0]
    else:
        data = data[:target_channels, :]

    window_size = int(WINDOW_SECONDS * target_sfreq)
    step_size = int((WINDOW_SECONDS - OVERLAP_SECONDS) * target_sfreq)

    for start in range(0, data.shape[1] - window_size + 1, step_size):
        window = data[:, start:start + window_size]

        scaler = StandardScaler()
        window = scaler.fit_transform(window.T).T

        X_windows.append(window)
        y_windows.append(row.label)

X = np.array(X_windows)
y = np.array(y_windows)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Labels:", np.bincount(y.astype(int)))

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
X shape: (625, 20, 512)
y shape: (625,)
Labels: [500 125]


In [6]:
from sklearn.model_selection import train_test_split

X = np.transpose(X, (0, 2, 1))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(500, 512, 20) (125, 512, 20)


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential()

model.add(Conv1D(32, 7, activation="relu", input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(Conv1D(64, 5, activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(Conv1D(128, 3, activation="relu"))
model.add(BatchNormalization())

model.add(Flatten())
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(64, activation="relu", name="feature_layer"))
model.add(Dense(1, activation="sigmoid"))

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 506, 32)        │         4,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 506, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 253, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 249, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 249, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 124, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 122, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 122, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 15616)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,998,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_layer (Dense)           │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,047,713 (7.81 MB)

 Trainable params: 2,047,265 (7.81 MB)

 Non-trainable params: 448 (1.75 KB)

In [8]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 8s 116ms/step - accuracy: 0.7000 - loss: 1.0412 - val_accuracy: 0.6100 - val_loss: 0.6851
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step - accuracy: 0.8625 - loss: 0.3867 - val_accuracy: 0.5500 - val_loss: 0.6747
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.8825 - loss: 0.3690 - val_accuracy: 0.5000 - val_loss: 0.7757
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accuracy: 0.9225 - loss: 0.2350 - val_accuracy: 0.5900 - val_loss: 0.8178
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - accuracy: 0.9525 - loss: 0.1749 - val_accuracy: 0.5700 - val_loss: 0.8323
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - accuracy: 0.9650 - loss: 0.0954 - val_accuracy: 0.6500 - val_loss: 0.7466
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.9750 - loss: 0.0981 - val_accuracy: 0.6500 - val_loss: 0.7546


In [9]:
loss, accuracy = model.evaluate(X_test, y_test)
print("CNN Accuracy:", accuracy)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5760 - loss: 0.7227
CNN Accuracy: 0.5759999752044678


In [10]:
from tensorflow.keras.models import Model
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

feature_model = Model(
    inputs=model.inputs,
    outputs=model.get_layer("feature_layer").output
)

train_features = feature_model.predict(X_train)
test_features = feature_model.predict(X_test)

svm = SVC(kernel="linear")
svm.fit(train_features, y_train)

y_pred = svm.predict(test_features)

print("SVM Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Eyes_Closed", "Task"]))
print(confusion_matrix(y_test, y_pred))

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
SVM Accuracy: 0.712
              precision    recall  f1-score   support

 Eyes_Closed       0.80      0.86      0.83       100
        Task       0.18      0.12      0.14        25

    accuracy                           0.71       125
   macro avg       0.49      0.49      0.48       125
weighted avg       0.67      0.71      0.69       125

[[86 14]
 [22  3]]
